In [1]:
import pandas as pd 

pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 120) 
pd.set_option("display.max_columns", 100)

In [2]:
df_hsv = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/hsv_features.csv", skipinitialspace=True) 
df_mhi = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/mhi_features.csv", skipinitialspace=True) 
df_glcm = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/glcm_features.csv", skipinitialspace=True) 
df_flow = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/flow_features.csv", skipinitialspace=True) 
df_hog = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/new_hog.csv", skipinitialspace=True) 
df_metadata = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/metadata.csv", skipinitialspace=True) 

for df in [df_mhi, df_metadata, df_flow, df_glcm, df_hsv, df_hog]: 
    df.columns = df.columns.str.strip() 

for df in [df_mhi, df_metadata, df_flow, df_glcm, df_hsv, df_hog]: 
    df.columns = df.columns.str.strip() 
    for col in df.select_dtypes(include=['object', 'string']).columns:
        df[col] = df[col].astype(str).str.strip()

In [3]:
from pathlib import Path 
import cv2 
def extract_metadata(video_path) -> dict: 
        video_path = Path(video_path)
        cap = cv2.VideoCapture(str(video_path)) 
        if not cap.isOpened(): 
            print(f"[Error]: {video_path}")
            return None  
        metadata = {} 
        metadata["uuid"] = video_path.stem 
        metadata["filename"] = video_path.name 
        metadata["filepath"] = str(video_path) 
        metadata["category"] = video_path.parent.name 
        metadata["fps"] = cap.get(cv2.CAP_PROP_FPS) 
        metadata["total_frames"] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) 
        metadata["duration"] = metadata["total_frames"] / metadata["fps"]
        metadata["width"] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        metadata["height"] = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release() 
        return metadata
mtdt = extract_metadata("/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4") 
print(mtdt) 

{'uuid': 'dog_swimming_41', 'filename': 'dog_swimming_41.mp4', 'filepath': '/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4', 'category': 'swimming', 'fps': 29.98955668901636, 'total_frames': 258, 'duration': 8.602994791666667, 'width': 576, 'height': 1024}


In [4]:
new_row = {
    'uuid': 'dog_swimming_41',
    'filename': 'dog_swimming_41.mp4',
    'filepath': '/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4',
    'category': 'swimming',
    'fps': 29.98955668901636,
    'total_frames': 258,
    'duration': 8.602994791666667,
    'width': 576,
    'height': 1024
}

df_metadata = pd.concat(
    [df_metadata, pd.DataFrame([new_row])],
    ignore_index=True
)

In [5]:
def normalize_video_name(value):
    if pd.isna(value):
        return value

    return str(value).replace(
        "dog_eat_drink_",
        "dog_eat_"
    )
df_mhi["path"] = df_mhi["path"].apply(normalize_video_name) 
df_hog["filepath"] = df_hog["filepath"].apply(normalize_video_name) 
df_flow["path"] = df_flow["path"].apply(normalize_video_name) 

In [6]:
import os 
def get_filename(path):
    if pd.isna(path):
        return None

    path = str(path).strip()

    # Chuẩn hóa dấu phân cách đường dẫn
    path = path.replace("\\", "/")

    return os.path.basename(path)

In [7]:
df_metadata["filename"] = (
    df_metadata["filename"]
    .astype(str)
    .str.strip()
)

df_metadata["filepath"] = (
    df_metadata["filepath"]
    .astype(str)
    .str.strip()
)

df_metadata["filename"] = df_metadata["filename"].apply(get_filename)
df_hsv["filename"] = df_hsv["video_path"].apply(get_filename)
df_glcm["filename"] = df_glcm["video_path"].apply(get_filename)
df_flow["filename"] = df_flow["path"].apply(get_filename)
df_mhi["filename"] = df_mhi["path"].apply(get_filename)
df_hog["filename"] = df_hog["filepath"].apply(get_filename)

In [8]:
# Thực hiện thay thế trực tiếp vào cột filename của df_mhi
df_hog['filename'] = (
    df_hog['filename']
    .astype(str)
    .str.replace('dog_lying', 'dog_lie', regex=False)  # Đổi dog_lying -> dog_lie
    .str.replace(r'_0(\d)', r'_\1', regex=True)       # Đổi _01, _02... thành _1, _2...
)

# Thực hiện thay thế trực tiếp vào cột filename của df_mhi
df_mhi['filename'] = (
    df_mhi['filename']
    .astype(str)
    .str.replace('dog_lying', 'dog_lie', regex=False)  # Đổi dog_lying -> dog_lie
    .str.replace(r'_0(\d)', r'_\1', regex=True)       # Đổi _01, _02... thành _1, _2...
)

# Thực hiện thay thế trực tiếp vào cột filename của df_mhi
df_flow['filename'] = (
    df_flow['filename']
    .astype(str)
    .str.replace('dog_lying', 'dog_lie', regex=False)  # Đổi dog_lying -> dog_lie
    .str.replace(r'_0(\d)', r'_\1', regex=True)       # Đổi _01, _02... thành _1, _2...
)

In [9]:
# Áp dụng cho tất cả các dòng trong cột filename
df_mhi['filename'] = (
    df_mhi['filename']
    .astype(str)
    .str.replace(
        r'_(\d)(?!\d)', 
        lambda m: f'_0{m.group(1)}', 
        regex=True
    )
) 
# Áp dụng cho tất cả các dòng trong cột filename
df_hog['filename'] = (
    df_hog['filename']
    .astype(str)
    .str.replace(
        r'_(\d)(?!\d)', 
        lambda m: f'_0{m.group(1)}', 
        regex=True
    )
)
# Áp dụng cho tất cả các dòng trong cột filename
df_flow['filename'] = (
    df_flow['filename']
    .astype(str)
    .str.replace(
        r'_(\d)(?!\d)', 
        lambda m: f'_0{m.group(1)}', 
        regex=True
    )
)
# Áp dụng cho tất cả các dòng trong cột filename
df_hsv['filename'] = (
    df_hsv['filename']
    .astype(str)
    .str.replace(
        r'_(\d)(?!\d)', 
        lambda m: f'_0{m.group(1)}', 
        regex=True
    )
)
# Áp dụng cho tất cả các dòng trong cột filename
df_glcm['filename'] = (
    df_glcm['filename']
    .astype(str)
    .str.replace(
        r'_(\d)(?!\d)', 
        lambda m: f'_0{m.group(1)}', 
        regex=True
    )
)
# Áp dụng cho tất cả các dòng trong cột filename
df_metadata['filename'] = (
    df_metadata['filename']
    .astype(str)
    .str.replace(
        r'_(\d)(?!\d)', 
        lambda m: f'_0{m.group(1)}', 
        regex=True
    )
)

In [10]:
datasets = {
    "HSV": df_hsv,
    "GLCM": df_glcm,
    "HOG": df_hog,
    "Flow": df_flow,
    "MHI": df_mhi, 
    "METADATA": df_metadata 
}
filename_sets = {
    name: set(df["filename"].dropna().unique())
    for name, df in datasets.items()
}
# Lấy phần giao của cả 5 tập hợp
common_filenames = set.intersection(*filename_sets.values())
print("Số video chung:", len(common_filenames))
for name in common_filenames: 
    print(name) 

Số video chung: 501
dog_eat_41.mp4
dog_standing_21.mp4
dog_running_02.mp4
dog_running_80.mp4
dog_lie_17.mp4
dog_walking_70.mp4
dog_jumping_21.mp4
dog_jumping_13.mp4
dog_lie_43.mp4
dog_running_14.mp4
dog_walking_11.mp4
dog_sitting_06.mp4
dog_swimming_20.mp4
dog_swimming_35.mp4
dog_lie_10.mp4
dog_running_19.mp4
dog_sitting_31.mp4
dog_swimming_32.mp4
dog_jumping_55.mp4
dog_jumping_07.mp4
dog_standing_15.mp4
dog_running_44.mp4
dog_running_36.mp4
dog_eat_43.mp4
dog_running_08.mp4
dog_lie_47.mp4
dog_lie_16.mp4
dog_running_81.mp4
dog_standing_08.mp4
dog_walking_32.mp4
dog_walking_20.mp4
dog_running_04.mp4
dog_walking_27.mp4
dog_sitting_18.mp4
dog_jumping_47.mp4
dog_walking_53.mp4
dog_running_48.mp4
dog_swimming_33.mp4
dog_eat_67.mp4
dog_running_54.mp4
dog_lie_27.mp4
dog_running_28.mp4
dog_walking_24.mp4
dog_lie_33.mp4
dog_eat_15.mp4
dog_walking_31.mp4
dog_standing_13.mp4
dog_eat_08.mp4
dog_sitting_21.mp4
dog_sitting_23.mp4
dog_running_67.mp4
dog_sitting_09.mp4
dog_sitting_19.mp4
dog_running_9

In [11]:
df_hsv = df_hsv[df_hsv["filename"].isin(common_filenames)].copy()
df_glcm = df_glcm[df_glcm["filename"].isin(common_filenames)].copy()
df_hog = df_hog[df_hog["filename"].isin(common_filenames)].copy()
df_flow = df_flow[df_flow["filename"].isin(common_filenames)].copy()
df_mhi = df_mhi[df_mhi["filename"].isin(common_filenames)].copy()
df_metadata = df_metadata[df_metadata["filename"].isin(common_filenames)].copy()

In [12]:
df_metadata = df_metadata.reset_index(drop=True)

df_metadata.insert(
    0,
    "video_id",
    range(1, len(df_metadata) + 1)
)

In [13]:
video_mapping = df_metadata[
    ["video_id", "filename", "filepath"]
].copy() 

video_mapping

,video_id,filename,filepath
0,1,dog_eat_01.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4
1,2,dog_eat_20.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_20.mp4
2,3,dog_eat_66.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_66.mp4
3,4,dog_eat_54.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_54.mp4
4,5,dog_eat_59.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_59.mp4
...,...,...,...
496,497,dog_standing_23.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_23.mp4
497,498,dog_standing_24.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_24.mp4
498,499,dog_standing_03.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_03.mp4
499,500,dog_standing_25.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4


In [14]:
df_hsv = df_hsv.merge(
    video_mapping,
    on="filename",
    how="left",
    validate="many_to_one"
)

df_hsv = df_hsv.rename(
    columns={
        "timestamp": "timestamp_sec",
        "hsv_vector": "vector"
    }
)

df_hsv = df_hsv[
    [
        "video_id",
        "filepath",
        "frame_idx",
        "timestamp_sec",
        "vector"
    ]
]

df_hsv 

,video_id,filepath,frame_idx,timestamp_sec,vector
0,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,0,0.000000,"[0.02617924528301887, 0.15947572720125786, 0.14300560141509433, 0.004999508647798742, 0.0035598466981132076, 1.47405..."
1,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,30,1.002124,"[0.02809060534591195, 0.15074685534591195, 0.1630282036163522, 0.004456564465408805, 0.002793337264150943, 0.0, 0.00..."
2,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,60,2.004249,"[0.033814858490566035, 0.15899174528301888, 0.14856033805031446, 0.004925805817610063, 0.0030169025157232706, 3.1937..."
3,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,90,3.006373,"[0.029247739779874212, 0.17289946933962264, 0.1443224253144654, 0.00582498034591195, 0.0022675904088050314, 9.335691..."
4,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,120,4.008498,"[0.031257370283018865, 0.1859571540880503, 0.14530267295597485, 0.0070263364779874216, 0.005761104559748428, 0.00014..."
...,...,...,...,...,...
5864,500,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,160,5.066667,"[0.03300010175010175, 0.1827498600936101, 0.09560407763532763, 0.014410994098494098, 0.013188721001221001, 0.0005135..."
5865,500,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,192,6.080000,"[0.027061075498575497, 0.16922854344729343, 0.08970765923890923, 0.013386497761497761, 0.013176956145706145, 0.00028..."
5866,500,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,224,7.093333,"[0.0256044591982092, 0.16512197293447292, 0.08431712962962963, 0.013245637464387464, 0.011409366096866097, 0.0003100..."
5867,500,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,256,8.106667,"[0.026929118335368336, 0.15871267043142043, 0.0885423026048026, 0.012749923687423687, 0.011292989417989417, 0.000361..."


In [15]:
df_glcm = df_glcm.drop(columns=["video_id"]).merge(
    video_mapping,
    on="filename",
    how="left",
    validate="many_to_one"
)

df_glcm = df_glcm.rename(
    columns={
        "glcm_vector": "vector"
    }
)

df_glcm = df_glcm[
    [
        "video_id",
        "filepath",
        "frame_idx",
        "timestamp_sec",
        "vector"
    ]
]
df_glcm

,video_id,filepath,frame_idx,timestamp_sec,vector
0,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,0,0.000,"[0.11328129470348358, 0.31043195724487305, 3.679398536682129, 0.8967195749282837]"
1,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,30,1.002,"[0.10935035347938538, 0.31993210315704346, 3.7216877937316895, 0.8939757943153381]"
2,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,60,2.004,"[0.10772231966257095, 0.3149460256099701, 3.7200679779052734, 0.8960456252098083]"
3,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,90,3.006,"[0.1101420670747757, 0.2990172505378723, 3.6747984886169434, 0.8996459245681763]"
4,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,120,4.008,"[0.10790690034627914, 0.33149415254592896, 3.7249674797058105, 0.891003429889679]"
...,...,...,...,...,...
5864,500,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,160,5.067,"[0.13164307177066803, 0.2307320535182953, 3.4436392784118652, 0.9050720930099487]"
5865,500,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,192,6.080,"[0.13610300421714783, 0.2032986581325531, 3.379974365234375, 0.9137535691261292]"
5866,500,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,224,7.093,"[0.13256525993347168, 0.2116389274597168, 3.4144790172576904, 0.9104992151260376]"
5867,500,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,256,8.107,"[0.13607095181941986, 0.1933104395866394, 3.3655920028686523, 0.9156087636947632]"


In [16]:
df_flow = df_flow.drop(columns=["video_id"]).merge(
    video_mapping,
    on="filename",
    how="left",
    validate="many_to_one"
)

df_flow = df_flow.rename(
    columns={
        "optical_flow_vector": "vector"
    }
)

df_flow = df_flow[
    [
        "video_id",
        "filepath",
        "vector"
    ]
]
df_flow

,video_id,filepath,vector
0,31,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_4.mp4,"[0.1894336640834808, 0.0538028478622436, 0.0538028478622436, 0.1894336640834808, 0.2074091434478759, 0.0493543334305..."
1,66,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_2.mp4,"[0.1529983878135681, 0.0856940448284149, 0.0856940448284149, 0.1529983878135681, 0.1676016598939895, 0.0937059000134..."
2,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,"[0.2318705320358276, 0.0241991709917783, 0.0241991709917783, 0.2318705320358276, 0.1829137951135635, 0.0610164999961..."
3,48,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_3.mp4,"[0.1781081855297088, 0.071504071354866, 0.071504071354866, 0.1781081855297088, 0.1882883012294769, 0.062099438160657..."
4,7,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_5.mp4,"[0.1747577488422393, 0.0483232736587524, 0.0483232736587524, 0.1747577488422393, 0.1945402026176452, 0.0823787823319..."
...,...,...,...
496,135,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_73.mp4,"[0.0051622251048684, 0.002950916532427, 0.002950916532427, 0.0051622251048684, 0.3577311635017395, 0.134155690670013..."
497,90,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_76.mp4,"[0.1553722918033599, 0.0869016274809837, 0.0869016274809837, 0.1553722918033599, 0.2091047912836074, 0.0486212819814..."
498,75,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_71.mp4,"[0.2472555935382843, 0.2010935395956039, 0.2010935395956039, 0.2472555935382843, 0.0373349338769912, 0.0143159236758..."
499,124,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_77.mp4,"[0.1587321460247039, 0.0049533485434949, 0.0049533485434949, 0.1587321460247039, 0.3298554122447967, 0.0064590908586..."


In [17]:
# Đồng bộ với metadata
df_mhi = df_mhi.drop(columns=["video_id"]).merge(
    video_mapping,
    on="filename",
    how="left",
    validate="many_to_one"
)

df_mhi = df_mhi.rename(
    columns={
        "mhi_vector": "vector"
    }
)

df_mhi = df_mhi[
    [
        "video_id",
        "filepath",
        "mhi_window_index",
        "vector"
    ]
]

df_mhi 

,video_id,filepath,mhi_window_index,vector
0,31,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_4.mp4,0,"[0.0, 0.0533333346247673, 0.4250000119209289, 0.4875000119209289, 0.2191666662693023, 0.0608333349227905, 0.06500000..."
1,31,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_4.mp4,1,"[0.0, 0.0, 0.0766666680574417, 0.2600000202655792, 0.3041666746139526, 0.1108333393931388, 0.0041666668839752, 0.063..."
2,31,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_4.mp4,2,"[0.0, 0.0, 0.0199999995529651, 0.2425000071525573, 0.2083333432674408, 0.1599999964237213, 0.0983333364129066, 0.168..."
3,31,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_4.mp4,3,"[0.7200000286102295, 0.9991666674613952, 0.912500023841858, 0.8075000047683716, 0.9133333563804626, 0.84750002622604..."
4,31,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_4.mp4,4,"[0.5550000071525574, 0.4783333539962768, 0.4408333301544189, 0.4558333456516266, 0.940833330154419, 0.54583334922790..."
...,...,...,...,...
5734,145,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_69.mp4,14,"[0.2199999988079071, 0.4691666662693023, 0.2341666668653488, 0.3516666889190674, 0.5074999928474426, 0.6058333516120..."
5735,145,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_69.mp4,15,"[0.1416666656732559, 0.5, 0.4141666889190674, 0.1133333370089531, 0.6075000166893005, 0.6675000190734863, 0.50499999..."
5736,145,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_69.mp4,16,"[0.1816666722297668, 0.4208333492279053, 0.5041666626930237, 0.1533333361148834, 0.51583331823349, 0.741666674613952..."
5737,145,/home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_69.mp4,17,"[0.2400000095367431, 0.2183333337306976, 0.5200000405311584, 0.1875, 0.2041666656732559, 0.7091667056083679, 0.53750..."


In [18]:
df_hog =  df_hog.merge(
    video_mapping,
    on="filename",
    how="left",
    validate="many_to_one"
)

df_hog = df_hog.rename(
    columns={
        "timestamp_seconds": "timestamp_sec",
        "hog_vector": "vector", 
        "frame_index": "frame_idx",
        "filepath_x": "filepath"
    }
)

df_hog = df_hog[
    [
        "video_id",
        "filepath",
        "frame_idx",
        "timestamp_sec",
        "vector"
    ]
]

df_hog

,video_id,filepath,frame_idx,timestamp_sec,vector
0,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,0,0.000000,"[0.0,0.0,0.025137953460216522,0.024710429832339287,0.4919843375682831,0.07201147079467773,0.03922506421804428,0.1192..."
1,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,30,1.002124,"[0.06466596573591232,0.0,0.010161276906728745,0.060379453003406525,0.4571734666824341,0.06621473282575607,0.10998835..."
2,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,60,2.004249,"[0.04622513800859451,0.0,0.048020973801612854,0.04975811392068863,0.45255112648010254,0.0597614049911499,0.070828504..."
3,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,90,3.006373,"[0.12181007862091064,0.0,0.02088066190481186,0.06319819390773773,0.46710577607154846,0.0515255443751812,0.0920552834..."
4,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,120,4.008498,"[0.08931413292884827,0.0,0.050257980823516846,0.016642706468701363,0.48089322447776794,0.06254532933235168,0.0718306..."
...,...,...,...,...,...
5864,500,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,160,5.066667,"[0.12014636397361755,0.044375840574502945,0.09869328886270523,0.08621378242969513,0.17860931158065796,0.229044631123..."
5865,500,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,192,6.080000,"[0.06277672946453094,0.0,0.0,0.01437972579151392,0.09445373713970184,0.10067149996757507,0.10244996845722198,0.03792..."
5866,500,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,224,7.093333,"[0.11207787692546844,0.0,0.052626609802246094,0.08808694779872894,0.2692693769931793,0.04471343010663986,0.032055970..."
5867,500,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,256,8.106667,"[0.17751573026180267,0.02304021455347538,0.00806760136038065,0.05253395810723305,0.27667298913002014,0.1062609627842..."


In [19]:
df_metadata = df_metadata.sort_values(by="video_id", ascending=True) 
df_hsv = df_hsv.sort_values(by=["video_id", "timestamp_sec"], ascending=[True, True]) 
df_glcm = df_glcm.sort_values(by=["video_id", "timestamp_sec"], ascending=[True, True]) 
df_hog = df_hog.sort_values(by=["video_id", "timestamp_sec"], ascending=[True, True]) 
df_mhi = df_mhi.sort_values(by=["video_id", "mhi_window_index"], ascending=[True, True]) 
df_flow = df_flow.sort_values(by="video_id", ascending=True) 

print("="*50 + "METADATA" + "="*50) 
display(df_metadata) 
print("="*50 + "HSV" + "="*50) 
display(df_hsv) 
print("="*50 + "GLCM" + "="*50) 
display(df_glcm) 
print("="*50 + "MHI" + "="*50) 
display(df_mhi) 
print("="*50 + "FLOW" + "="*50) 
display(df_flow) 
print("="*50 + "HOG" + "="*50) 
display(df_hog) 

==================================================METADATA==================================================


,video_id,uuid,filename,filepath,category,fps,total_frames,duration,width,height
0,1,dog_eat_1,dog_eat_01.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,Eating,29.936401,466,15.566333,848,480
1,2,dog_eat_20,dog_eat_20.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_20.mp4,Eating,24.000000,129,5.375000,720,1280
2,3,dog_eat_66,dog_eat_66.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_66.mp4,Eating,24.000000,325,13.541667,2732,1440
3,4,dog_eat_54,dog_eat_54.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_54.mp4,Eating,60.000000,1124,18.733333,2160,3840
4,5,dog_eat_59,dog_eat_59.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_59.mp4,Eating,25.000000,293,11.720000,1920,1080
...,...,...,...,...,...,...,...,...,...,...
496,497,dog_standing_23,dog_standing_23.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_23.mp4,Stand,29.970030,199,6.639967,3840,2160
497,498,dog_standing_24,dog_standing_24.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_24.mp4,Stand,23.976024,299,12.470792,2160,3840
498,499,dog_standing_03,dog_standing_03.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_03.mp4,Stand,25.000000,240,9.600000,2160,3840
499,500,dog_standing_25,dog_standing_25.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,Stand,31.578947,320,10.133333,1440,2184


==================================================HSV==================================================


,video_id,filepath,frame_idx,timestamp_sec,vector
0,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,0,0.000000,"[0.02617924528301887, 0.15947572720125786, 0.14300560141509433, 0.004999508647798742, 0.0035598466981132076, 1.47405..."
1,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,30,1.002124,"[0.02809060534591195, 0.15074685534591195, 0.1630282036163522, 0.004456564465408805, 0.002793337264150943, 0.0, 0.00..."
2,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,60,2.004249,"[0.033814858490566035, 0.15899174528301888, 0.14856033805031446, 0.004925805817610063, 0.0030169025157232706, 3.1937..."
3,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,90,3.006373,"[0.029247739779874212, 0.17289946933962264, 0.1443224253144654, 0.00582498034591195, 0.0022675904088050314, 9.335691..."
4,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,120,4.008498,"[0.031257370283018865, 0.1859571540880503, 0.14530267295597485, 0.0070263364779874216, 0.005761104559748428, 0.00014..."
...,...,...,...,...,...
3562,501,/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4,30,1.000348,"[0.008673773871527778, 0.034798516167534724, 0.014607747395833334, 0.0007188585069444445, 0.10388692220052083, 0.010..."
3563,501,/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4,60,2.000696,"[0.0102996826171875, 0.0818447536892361, 0.011311848958333334, 0.0008104112413194445, 0.08101569281684028, 0.0014326..."
3564,501,/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4,90,3.001045,"[0.013331095377604166, 0.07685173882378472, 0.010411580403645834, 0.0034434000651041665, 0.08357408311631945, 0.0002..."
3565,501,/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4,120,4.001393,"[0.012378268771701388, 0.060221354166666664, 0.010255601671006944, 0.004377577039930556, 0.08312310112847222, 0.0006..."


==================================================GLCM==================================================


,video_id,filepath,frame_idx,timestamp_sec,vector
0,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,0,0.000,"[0.11328129470348358, 0.31043195724487305, 3.679398536682129, 0.8967195749282837]"
1,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,30,1.002,"[0.10935035347938538, 0.31993210315704346, 3.7216877937316895, 0.8939757943153381]"
2,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,60,2.004,"[0.10772231966257095, 0.3149460256099701, 3.7200679779052734, 0.8960456252098083]"
3,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,90,3.006,"[0.1101420670747757, 0.2990172505378723, 3.6747984886169434, 0.8996459245681763]"
4,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,120,4.008,"[0.10790690034627914, 0.33149415254592896, 3.7249674797058105, 0.891003429889679]"
...,...,...,...,...,...
3562,501,/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4,30,1.000,"[0.2876371741294861, 0.10264227539300919, 2.389775276184082, 0.958886444568634]"
3563,501,/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4,60,2.001,"[0.2683045268058777, 0.1236317902803421, 2.4755868911743164, 0.9512495994567871]"
3564,501,/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4,90,3.001,"[0.29524704813957214, 0.14005054533481598, 2.442166328430176, 0.9482483863830566]"
3565,501,/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4,120,4.001,"[0.2603105902671814, 0.15878720581531525, 2.5878829956054688, 0.9419488310813904]"


... (2 outputs left out)

==================================================FLOW==================================================


,video_id,filepath,vector
2,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,"[0.2318705320358276, 0.0241991709917783, 0.0241991709917783, 0.2318705320358276, 0.1829137951135635, 0.0610164999961..."
18,2,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_20.mp4,"[0.2141327559947967, 0.0366533733904361, 0.0366533733904361, 0.2141327559947967, 0.2329324036836624, 0.0162814706563..."
64,3,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_66.mp4,"[0.2740306556224823, 0.0738999918103218, 0.0738999918103218, 0.2740306556224823, 0.10655078291893, 0.045518580824136..."
63,4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_54.mp4,"[0.2108207941055297, 0.0207749251276254, 0.0207749251276254, 0.2108207941055297, 0.2436993420124054, 0.0247049406170..."
57,5,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_59.mp4,"[0.081211507320404, 0.1795071512460708, 0.1795071512460708, 0.081211507320404, 0.0669059306383133, 0.172375410795211..."
...,...,...,...
365,497,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_23.mp4,"[0.2019945681095123, 0.1160381659865379, 0.1160381659865379, 0.2019945681095123, 0.0810988843441009, 0.1008683741092..."
368,498,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_24.mp4,"[0.2816620469093323, 0.0032428367994725, 0.0032428367994725, 0.2816620469093323, 0.1954365372657776, 0.0196585655212..."
346,499,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_03.mp4,"[0.173663780093193, 0.0467685423791408, 0.0467685423791408, 0.173663780093193, 0.2296632230281829, 0.049904443323612..."
367,500,/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_25.mp4,"[0.2657096683979034, 0.0224111173301935, 0.0224111173301935, 0.2657096683979034, 0.180963933467865, 0.03091528080403..."


==================================================HOG==================================================


,video_id,filepath,frame_idx,timestamp_sec,vector
0,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,0,0.000000,"[0.0,0.0,0.025137953460216522,0.024710429832339287,0.4919843375682831,0.07201147079467773,0.03922506421804428,0.1192..."
1,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,30,1.002124,"[0.06466596573591232,0.0,0.010161276906728745,0.060379453003406525,0.4571734666824341,0.06621473282575607,0.10998835..."
2,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,60,2.004249,"[0.04622513800859451,0.0,0.048020973801612854,0.04975811392068863,0.45255112648010254,0.0597614049911499,0.070828504..."
3,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,90,3.006373,"[0.12181007862091064,0.0,0.02088066190481186,0.06319819390773773,0.46710577607154846,0.0515255443751812,0.0920552834..."
4,1,/home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4,120,4.008498,"[0.08931413292884827,0.0,0.050257980823516846,0.016642706468701363,0.48089322447776794,0.06254532933235168,0.0718306..."
...,...,...,...,...,...
3562,501,/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4,30,1.000348,"[0.31485870480537415,0.0,0.009858179837465286,0.02216421812772751,0.11501797288656235,0.04081297665834427,0.04214130..."
3563,501,/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4,60,2.000696,"[0.2763010561466217,0.017456036061048508,0.024102559313178062,0.00581867853179574,0.07025919109582901,0.019866211339..."
3564,501,/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4,90,3.001045,"[0.2079683095216751,0.09676885604858398,0.012656999751925468,0.0,0.12657000124454498,0.024931399151682854,0.13835941..."
3565,501,/home/ninhdao/D23_Lop4_Nhom16/data_collection/swimming/dog_swimming_41.mp4,120,4.001393,"[0.16182248294353485,0.0,0.0,0.0,0.06428564339876175,0.017871981486678123,0.31815868616104126,0.31815868616104126,0...."


In [21]:
import pandas as pd

# =====================================================
# 1. Kiểm tra shape và cột
# =====================================================
for name, df in [("HSV", df_hsv), ("GLCM", df_glcm), ("HOG", df_hog)]:
    print(f"{name}: shape={df.shape}, columns={df.columns.tolist()}")
    print(f"  video_id unique: {df['video_id'].nunique()}")
    print(f"  video_id range : {df['video_id'].min()} -> {df['video_id'].max()}")

# =====================================================
# 2. Kiểm tra trùng lặp (video_id, frame_idx)
# =====================================================
for name, df in [("HSV", df_hsv), ("GLCM", df_glcm), ("HOG", df_hog)]:
    n_dup = df.duplicated(subset=["video_id", "frame_idx"]).sum()
    print(f"{name}: số dòng trùng (video_id, frame_idx) = {n_dup}")

# =====================================================
# 3. So sánh tập (video_id, frame_idx) giữa 3 bảng
# =====================================================
keys_hsv  = set(zip(df_hsv["video_id"],  df_hsv["frame_idx"]))
keys_glcm = set(zip(df_glcm["video_id"], df_glcm["frame_idx"]))
keys_hog  = set(zip(df_hog["video_id"],  df_hog["frame_idx"]))

print(f"\nSố key duy nhất:")
print(f"  HSV : {len(keys_hsv)}")
print(f"  GLCM: {len(keys_glcm)}")
print(f"  HOG : {len(keys_hog)}")

print(f"\nGiao 3 tập:")
print(f"  HSV ∩ GLCM          : {len(keys_hsv & keys_glcm)}")
print(f"  HSV ∩ HOG           : {len(keys_hsv & keys_hog)}")
print(f"  GLCM ∩ HOG          : {len(keys_glcm & keys_hog)}")
print(f"  HSV ∩ GLCM ∩ HOG    : {len(keys_hsv & keys_glcm & keys_hog)}")

# =====================================================
# 4. Key nào có trong HSV/GLCM nhưng THIẾU ở HOG
# =====================================================
missing_in_hog = (keys_hsv & keys_glcm) - keys_hog
print(f"\nKey có ở HSV+GLCM nhưng THIẾU ở HOG: {len(missing_in_hog)}")
if missing_in_hog:
    print("  20 key đầu:", sorted(missing_in_hog)[:20])

missing_in_hsv = (keys_glcm & keys_hog) - keys_hsv
print(f"Key có ở GLCM+HOG nhưng THIẾU ở HSV: {len(missing_in_hsv)}")
if missing_in_hsv:
    print("  20 key đầu:", sorted(missing_in_hsv)[:20])

missing_in_glcm = (keys_hsv & keys_hog) - keys_glcm
print(f"Key có ở HSV+HOG nhưng THIẾU ở GLCM: {len(missing_in_glcm)}")
if missing_in_glcm:
    print("  20 key đầu:", sorted(missing_in_glcm)[:20])

HSV: shape=(5869, 5), columns=['video_id', 'filepath', 'frame_idx', 'timestamp_sec', 'vector']
  video_id unique: 501
  video_id range : 1 -> 501
GLCM: shape=(5869, 5), columns=['video_id', 'filepath', 'frame_idx', 'timestamp_sec', 'vector']
  video_id unique: 501
  video_id range : 1 -> 501
HOG: shape=(5851, 5), columns=['video_id', 'filepath', 'frame_idx', 'timestamp_sec', 'vector']
  video_id unique: 501
  video_id range : 1 -> 501
HSV: số dòng trùng (video_id, frame_idx) = 0
GLCM: số dòng trùng (video_id, frame_idx) = 0
HOG: số dòng trùng (video_id, frame_idx) = 0

Số key duy nhất:
  HSV : 5869
  GLCM: 5869
  HOG : 5851

Giao 3 tập:
  HSV ∩ GLCM          : 5869
  HSV ∩ HOG           : 5547
  GLCM ∩ HOG          : 5547
  HSV ∩ GLCM ∩ HOG    : 5547

Key có ở HSV+GLCM nhưng THIẾU ở HOG: 322
  20 key đầu: [(126, 25), (126, 50), (126, 75), (126, 100), (126, 125), (126, 150), (126, 175), (126, 200), (126, 225), (126, 250), (126, 275), (126, 300), (126, 325), (126, 350), (126, 375), (126,

In [20]:
keyframes = pd.concat(
    [
        df_hsv[
            ["video_id", "frame_idx", "timestamp_sec"]
        ], 
        df_glcm[
            ["video_id", "frame_idx", "timestamp_sec"]
        ], 
        df_hog[
            ["video_id", "frame_idx", "timestamp_sec"]
        ]
    ], 
    ignore_index=True 
) 

# Loại bỏ các keyframe trùng nhau
keyframes = keyframes.drop_duplicates(
    subset=["video_id", "frame_idx"]
)

# Tạo keyframe_id
keyframes.insert(
    0,
    "keyframe_id",
    range(1, len(keyframes) + 1)
)

# Tạo keyframe_index theo từng video
keyframes["keyframe_index"] = (
    keyframes.groupby("video_id").cumcount()
)

keyframes = keyframes.sort_values(by='keyframe_id') 
keyframes

,keyframe_id,video_id,frame_idx,timestamp_sec,keyframe_index
0,1,1,0,0.000000,0
1,2,1,30,1.002124,1
2,3,1,60,2.004249,2
3,4,1,90,3.006373,3
4,5,1,120,4.008498,4
...,...,...,...,...,...
5864,5865,501,30,1.000348,1
5865,5866,501,60,2.000696,2
5866,5867,501,90,3.001045,3
5867,5868,501,120,4.001393,4


In [21]:
df_hsv = df_hsv.merge(
    keyframes[
        ["keyframe_id", "video_id", "frame_idx"]
    ],
    on=["video_id", "frame_idx"],
    how="left",
    validate="one_to_one"
)

df_hsv = df_hsv[
    ["keyframe_id", "vector"]
]

df_hsv 

,keyframe_id,vector
0,1,"[0.02617924528301887, 0.15947572720125786, 0.14300560141509433, 0.004999508647798742, 0.0035598466981132076, 1.47405..."
1,2,"[0.02809060534591195, 0.15074685534591195, 0.1630282036163522, 0.004456564465408805, 0.002793337264150943, 0.0, 0.00..."
2,3,"[0.033814858490566035, 0.15899174528301888, 0.14856033805031446, 0.004925805817610063, 0.0030169025157232706, 3.1937..."
3,4,"[0.029247739779874212, 0.17289946933962264, 0.1443224253144654, 0.00582498034591195, 0.0022675904088050314, 9.335691..."
4,5,"[0.031257370283018865, 0.1859571540880503, 0.14530267295597485, 0.0070263364779874216, 0.005761104559748428, 0.00014..."
...,...,...
5864,5865,"[0.008673773871527778, 0.034798516167534724, 0.014607747395833334, 0.0007188585069444445, 0.10388692220052083, 0.010..."
5865,5866,"[0.0102996826171875, 0.0818447536892361, 0.011311848958333334, 0.0008104112413194445, 0.08101569281684028, 0.0014326..."
5866,5867,"[0.013331095377604166, 0.07685173882378472, 0.010411580403645834, 0.0034434000651041665, 0.08357408311631945, 0.0002..."
5867,5868,"[0.012378268771701388, 0.060221354166666664, 0.010255601671006944, 0.004377577039930556, 0.08312310112847222, 0.0006..."


In [22]:
df_glcm = df_glcm.merge(
    keyframes[
        ["keyframe_id", "video_id", "frame_idx"]
    ],
    on=["video_id", "frame_idx"],
    how="left",
    validate="one_to_one"
)

df_glcm = df_glcm[
    ["keyframe_id", "vector"]
] 
df_glcm

,keyframe_id,vector
0,1,"[0.11328129470348358, 0.31043195724487305, 3.679398536682129, 0.8967195749282837]"
1,2,"[0.10935035347938538, 0.31993210315704346, 3.7216877937316895, 0.8939757943153381]"
2,3,"[0.10772231966257095, 0.3149460256099701, 3.7200679779052734, 0.8960456252098083]"
3,4,"[0.1101420670747757, 0.2990172505378723, 3.6747984886169434, 0.8996459245681763]"
4,5,"[0.10790690034627914, 0.33149415254592896, 3.7249674797058105, 0.891003429889679]"
...,...,...
5864,5865,"[0.2876371741294861, 0.10264227539300919, 2.389775276184082, 0.958886444568634]"
5865,5866,"[0.2683045268058777, 0.1236317902803421, 2.4755868911743164, 0.9512495994567871]"
5866,5867,"[0.29524704813957214, 0.14005054533481598, 2.442166328430176, 0.9482483863830566]"
5867,5868,"[0.2603105902671814, 0.15878720581531525, 2.5878829956054688, 0.9419488310813904]"


In [23]:
df_hog = df_hog.merge(
    keyframes[
        ["keyframe_id", "video_id", "frame_idx"]
    ],
    on=["video_id", "frame_idx"],
    how="left",
    validate="one_to_one"
)

df_hog = df_hog[
    ["keyframe_id", "vector"]
]

df_hog

,keyframe_id,vector
0,1,"[0.0,0.0,0.025137953460216522,0.024710429832339287,0.4919843375682831,0.07201147079467773,0.03922506421804428,0.1192..."
1,2,"[0.06466596573591232,0.0,0.010161276906728745,0.060379453003406525,0.4571734666824341,0.06621473282575607,0.10998835..."
2,3,"[0.04622513800859451,0.0,0.048020973801612854,0.04975811392068863,0.45255112648010254,0.0597614049911499,0.070828504..."
3,4,"[0.12181007862091064,0.0,0.02088066190481186,0.06319819390773773,0.46710577607154846,0.0515255443751812,0.0920552834..."
4,5,"[0.08931413292884827,0.0,0.050257980823516846,0.016642706468701363,0.48089322447776794,0.06254532933235168,0.0718306..."
...,...,...
5864,5865,"[0.31485870480537415,0.0,0.009858179837465286,0.02216421812772751,0.11501797288656235,0.04081297665834427,0.04214130..."
5865,5866,"[0.2763010561466217,0.017456036061048508,0.024102559313178062,0.00581867853179574,0.07025919109582901,0.019866211339..."
5866,5867,"[0.2079683095216751,0.09676885604858398,0.012656999751925468,0.0,0.12657000124454498,0.024931399151682854,0.13835941..."
5867,5868,"[0.16182248294353485,0.0,0.0,0.0,0.06428564339876175,0.017871981486678123,0.31815868616104126,0.31815868616104126,0...."


In [24]:
df_metadata.to_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/processed_features/metadata.csv", index=False) 

In [25]:
df_hsv.to_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/processed_features/hsv_features.csv", index=False)

In [26]:
df_glcm.to_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/processed_features/glcm_features.csv", index=False) 

In [27]:
df_hog.to_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/processed_features/hog_features.csv", index=False) 

In [28]:
df_flow.to_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/processed_features/flow_features.csv", index=False)

In [29]:
df_mhi.to_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/processed_features/mhi_features.csv", index=False) 

In [30]:
keyframes.to_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/processed_features/keyframes.csv", index=False)